# Modeling

## 1. Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

### 1.1. Drive mount, paths and extraction

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/NoteVision')
RAW_DIR = PROJECT_DIR / 'data_raw'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
LOCAL_DIR = Path('/content/primus')
ARCHIVE = RAW_DIR / 'primus.tgz'
EXPECTED_SAMPLES = 87_678
SEED = 42

def count_samples(root):
    n = 0
    for pkg in ('package_aa', 'package_ab'):
        p = root / pkg
        if p.exists():
            n += sum(1 for d in p.iterdir() if d.is_dir() and not d.name.startswith('._'))
    return n

if count_samples(LOCAL_DIR) != EXPECTED_SAMPLES:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $LOCAL_DIR

n = count_samples(LOCAL_DIR)
print('Samples on disk:', n)
assert n == EXPECTED_SAMPLES, 'Incomplete extraction'

Mounted at /content/drive
Samples on disk: 87678


### 1.2. Preprocessing functions and metrics

In [3]:
IMG_HEIGHT = 128

def load_vocab(vocab_csv):
    """Return (tokens, token_to_id, blank_index). CTC blank = last index."""
    df = pd.read_csv(vocab_csv, dtype=str, keep_default_na=False)
    tokens = df['token'].tolist()
    return tokens, {t: i for i, t in enumerate(tokens)}, len(tokens)

def encode_labels(df, token_to_id, max_len):
    """Dense int32 array [N, max_len] (zero padded) and true lengths [N]."""
    labels = np.zeros((len(df), max_len), dtype=np.int32)
    lengths = np.zeros(len(df), dtype=np.int32)
    for i, seq in enumerate(df['label']):
        ids = [token_to_id[t] for t in seq.split()]
        labels[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return labels, lengths

def decode_ids(ids, tokens):
    return ' '.join(tokens[i] for i in ids)

def image_path(root, package, sid):
    return Path(root) / package / sid / f'{sid}.png'

# ... Deleted functions (notebook previo)

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_png(raw, channels=3) # palette (mode P) -> RGB
    gray = tf.image.rgb_to_grayscale(img) # uint8 [H, W, 1]
    h = tf.cast(tf.shape(gray)[0], tf.float32)
    w = tf.cast(tf.shape(gray)[1], tf.float32)
    new_w = tf.maximum(1, tf.cast(tf.round(w * IMG_HEIGHT / h), tf.int32))
    gray = tf.image.resize(tf.cast(gray, tf.float32), (IMG_HEIGHT, new_w),
                           method='bilinear', antialias=True)
    gray = tf.clip_by_value(gray, 0.0, 255.0)
    return 1.0 - gray / 255.0, new_w # ink = 1, background/padding = 0

def build_dataset(df, root, token_to_id, max_label_len, batch_size=32,
                  training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width,
                'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(
        batch_size,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)})
    return ds.prefetch(tf.data.AUTOTUNE)

from collections import Counter, defaultdict

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (ref[i-1] != hyp[j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + (ref[i-1] != hyp[j-1]):
            ops.append(('hit' if ref[i-1] == hyp[j-1] else 'sub', ref[i-1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(('del', ref[i-1]))
            i -= 1
        else:
            ops.append(('ins', None))
            j -= 1
    return d[n][m], ops

def evaluate_sequences(refs, hyps, families):
    tot_ref = tot_err = tot_err_shape = exact = 0
    per_fam = defaultdict(Counter)
    for r, h in zip(refs, hyps):
        dist, ops = align(r, h)
        tot_err += dist
        tot_ref += len(r)
        exact += int(r == h)
        tot_err_shape += align([families[i] for i in r], [families[i] for i in h])[0]
        for op, tok in ops:
            if tok is not None:
                per_fam[families[tok]][op] += 1
    metrics = {'cer': tot_err / tot_ref, 'seq_acc': exact / len(refs),
               'cer_shape': tot_err_shape / tot_ref,
               'n_sequences': len(refs), 'n_tokens': tot_ref}
    rows = []
    for fam, c in per_fam.items():
        support = c['hit'] + c['sub'] + c['del']
        rows.append({'family': fam, 'support': support, 'recall': c['hit'] / support})
    fam_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)
    return metrics, fam_df

# Sanity checks
_fam = ['a', 'a', 'b']
assert align([0, 1, 2], [0, 1, 2])[0] == 0
assert align([0, 1, 2], [0, 2])[0] == 1
assert align([0, 1, 2], [0, 1, 1])[0] == 1
_m, _ = evaluate_sequences([[0, 1, 2]], [[0, 0, 2]], _fam)
assert abs(_m['cer'] - 1/3) < 1e-9 and _m['cer_shape'] == 0.0 and _m['seq_acc'] == 0.0
print('Metric sanity checks passed.')

Metric sanity checks passed.


### 1.3. Arranque

In [4]:
parts = {s: pd.read_csv(ARTIFACTS_DIR / f'split_{s}.csv', dtype=str)
         for s in ('train', 'val', 'test')}
for d in parts.values():
    d['seq_len'] = d['seq_len'].astype(int)

tokens, token_to_id, BLANK_INDEX = load_vocab(ARTIFACTS_DIR / 'vocab_agnostic.csv')
cfg = json.loads((ARTIFACTS_DIR / 'preprocessing_config.json').read_text())
summ = json.loads((ARTIFACTS_DIR / 'split_summary.json').read_text())

# Guard against drift between notebooks
assert IMG_HEIGHT == cfg['img_height'] and BLANK_INDEX == cfg['blank_index']
assert {s: len(d) for s, d in parts.items()} == {s: summ[f'n_{s}'] for s in parts}
MAX_LABEL_LEN = cfg['max_label_len']
print('Bootstrap OK:', {s: len(d) for s, d in parts.items()})

Bootstrap OK: {'train': 70138, 'val': 8772, 'test': 8768}


## 2. GPU and Seed

In [5]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(SEED)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. CRNN Model

In [6]:
REDUCTION = 8 # horizontal downsampling
NUM_CLASSES = len(tokens) + 1 # 758 tokens + CTC blank (last index)
assert NUM_CLASSES == BLANK_INDEX + 1
families = [t.split('-')[0] for t in tokens]


class CRNN(keras.Model):
    def __init__(self, num_classes, lstm_units=128, dropout=0.25, **kw):
        super().__init__(**kw)
        filters = (32, 64, 128, 256)
        pools = ((2, 2), (2, 2), (2, 2), (2, 1)) # width /2 three times -> x8; last pool only in height
        self.convs = [layers.Conv2D(f, 3, padding='same', use_bias=False) for f in filters]
        self.bns = [layers.BatchNormalization() for _ in filters]
        self.pools = [layers.MaxPool2D(p) for p in pools]
        self.proj = layers.Dense(256, activation='relu')
        self.drop = layers.Dropout(dropout)
        self.rnn1 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.rnn2 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x, widths = inputs['image'], inputs['width']
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = pool(tf.nn.relu(bn(conv(x), training=training)))
        x = tf.transpose(x, [0, 2, 1, 3])
        s = tf.shape(x)
        x = tf.reshape(x, [s[0], s[1], x.shape[2] * x.shape[3]])
        x = self.drop(self.proj(x), training=training)
        logit_len = widths // REDUCTION
        mask = tf.sequence_mask(logit_len, maxlen=s[1]) # ignore padding in the BiLSTM
        x = self.rnn1(x, mask=mask, training=training)
        x = self.rnn2(self.drop(x, training=training), mask=mask, training=training)
        return self.classifier(self.drop(x, training=training)), logit_len


model = CRNN(NUM_CLASSES)

# Shape smoke test on one real batch
train_ds = build_dataset(parts['train'], LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                         batch_size=16, training=True, seed=SEED)

b = next(iter(train_ds))
logits, logit_len = model({'image': b['image'], 'width': b['width']}, training=False)
print('image:', tuple(b['image'].shape), '| logits:', tuple(logits.shape))
print('logit_len:', logit_len.numpy())
assert logits.shape[1] == b['image'].shape[2] // REDUCTION
assert logits.shape[-1] == NUM_CLASSES
assert bool(tf.reduce_all(logit_len <= logits.shape[1]))
assert bool(tf.reduce_all(logit_len >= b['label_length']))
print('\nTrainable params:', sum(int(np.prod(v.shape)) for v in model.trainable_variables))

image: (16, 128, 1567, 1) | logits: (16, 195, 759)
logit_len: [ 92 110 137 192 129 134 139 165 132 134 113  77 179 195  92 157]

Trainable params: 1896407


## 4. CTC Loss, Training Step and Greedy Decoding

In [12]:
def ctc_loss_fn(labels, label_len, logits, logit_len):
    loss = tf.nn.ctc_loss(labels=labels, logits=tf.cast(logits, tf.float32),
                          label_length=label_len, logit_length=logit_len,
                          logits_time_major=False, blank_index=BLANK_INDEX)
    return tf.reduce_mean(loss)

optimizer = keras.optimizers.Adam(1e-3, clipnorm=5.0)
optimizer.build(model.trainable_variables)


@tf.function(reduce_retracing=True)
def train_step(image, width, label, label_len):
    with tf.GradientTape() as tape:
        logits, logit_len = model({'image': image, 'width': width}, training=True)
        loss = ctc_loss_fn(label, label_len, logits, logit_len)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss


@tf.function(reduce_retracing=True)
def infer(image, width):
    return model({'image': image, 'width': width}, training=False)


def greedy_decode(logits, lengths, blank=BLANK_INDEX):
    """argmax per step -> collapse repeats -> drop blanks."""
    ids = np.argmax(logits, axis=-1)
    out = []
    for seq, n in zip(ids, lengths):
        seq = seq[:n]
        changed = np.concatenate([[True], seq[1:] != seq[:-1]])
        seq = seq[changed]
        out.append(seq[seq != blank].tolist())
    return out


def predict(batches):
    refs, hyps = [], []
    for b in batches:
        logits, logit_len = infer(b['image'], b['width'])
        hyps += greedy_decode(logits.numpy(), logit_len.numpy())
        lab, ln = b['label'].numpy(), b['label_length'].numpy()
        refs += [lab[i, :ln[i]].tolist() for i in range(len(ln))]
    return refs, hyps

## 5. Overfit test (64 samples)

In [15]:
small_df = parts['train'].sample(64, random_state=SEED)
small_batches = list(build_dataset(small_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                   batch_size=16, training=False))
EPOCHS = 400

for epoch in range(1, EPOCHS + 1):
    losses = [float(train_step(b['image'], b['width'], b['label'], b['label_length']))
              for b in small_batches]
    if epoch == 1 or epoch % 20 == 0:
        refs, hyps = predict(small_batches)
        m, _ = evaluate_sequences(refs, hyps, families)
        print(f"epoch {epoch:>3} | loss {np.mean(losses):9.3f} | "
              f"CER {m['cer']:.4f} | seq_acc {m['seq_acc']:.3f}")
        if m['seq_acc'] == 1.0:
            break

print('REF:', decode_ids(refs[0], tokens)[:160])
print('HYP:', decode_ids(hyps[0], tokens)[:160])

epoch   1 | loss   665.844 | CER 1.0000 | seq_acc 0.000
epoch  20 | loss   127.497 | CER 1.0000 | seq_acc 0.000
epoch  40 | loss   114.167 | CER 0.9686 | seq_acc 0.000
epoch  60 | loss   105.043 | CER 0.9442 | seq_acc 0.000
epoch  80 | loss    92.656 | CER 0.9587 | seq_acc 0.000
epoch 100 | loss    78.642 | CER 0.9343 | seq_acc 0.000
epoch 120 | loss    62.288 | CER 0.8784 | seq_acc 0.000
epoch 140 | loss    48.297 | CER 0.7912 | seq_acc 0.000
epoch 160 | loss    34.872 | CER 0.7469 | seq_acc 0.000
epoch 180 | loss    23.885 | CER 0.4212 | seq_acc 0.000
epoch 200 | loss    16.597 | CER 0.2693 | seq_acc 0.016
epoch 220 | loss    11.414 | CER 0.1059 | seq_acc 0.188
epoch 240 | loss     7.396 | CER 0.0797 | seq_acc 0.297
epoch 260 | loss     4.860 | CER 0.0326 | seq_acc 0.625
epoch 280 | loss     3.642 | CER 0.0372 | seq_acc 0.578
epoch 300 | loss     2.536 | CER 0.0134 | seq_acc 0.906
epoch 320 | loss     1.945 | CER 0.0145 | seq_acc 0.859
epoch 340 | loss     1.463 | CER 0.0128 | seq_ac

## 6. Short Real-Data Run

### 6.1. Width-Bucketed Pipeline

In [8]:
BOUNDS = [700, 900, 1100, 1300, 1500, 1800, 2200]
MAX_PIXELS = 24000   # batch_size * width budget per bucket (below the 16x2002 already tested)
BATCH_SIZES = [max(4, min(32, MAX_PIXELS // b)) for b in BOUNDS] + [6]
print('Batch size per width bucket:', BATCH_SIZES)


def build_dataset_bucketed(df, root, token_to_id, max_label_len, training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width, 'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.bucket_by_sequence_length(
        element_length_func=lambda e: e['width'],
        bucket_boundaries=BOUNDS, bucket_batch_sizes=BATCH_SIZES,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)})
    return ds.prefetch(tf.data.AUTOTUNE)

Batch size per width bucket: [32, 26, 21, 18, 16, 13, 10, 6]


### 6.2. Input Pipeline Speed

In [9]:
import time
from itertools import islice

short_train = parts['train'].sample(6000, random_state=SEED)
short_val = parts['val'].sample(1000, random_state=SEED)

ds_test = build_dataset_bucketed(short_train, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                 training=True, seed=SEED)
t0, n_img, n_b = time.time(), 0, 0
for b in islice(ds_test, 60):
    n_img += int(b['image'].shape[0]); n_b += 1
dt = time.time() - t0
print(f'{n_b} batches, {n_img} images in {dt:.1f}s -> {n_img/dt:.0f} img/s from the pipeline alone')

60 batches, 1066 images in 9.8s -> 109 img/s from the pipeline alone


### 6.3. Fresh Model and Short Training Run

In [10]:
tf.keras.utils.set_random_seed(SEED)
model = CRNN(NUM_CLASSES)
model({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})  # build weights
optimizer = keras.optimizers.Adam(1e-3, clipnorm=5.0)
optimizer.build(model.trainable_variables)

@tf.function(input_signature=[
    tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
    tf.TensorSpec([None], tf.int32),
    tf.TensorSpec([None, MAX_LABEL_LEN], tf.int32),
    tf.TensorSpec([None], tf.int32)])
def train_step(image, width, label, label_len):
    with tf.GradientTape() as tape:
        logits, logit_len = model({'image': image, 'width': width}, training=True)
        loss = ctc_loss_fn(label, label_len, logits, logit_len)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function(input_signature=[
    tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
    tf.TensorSpec([None], tf.int32)])
def infer(image, width):
    return model({'image': image, 'width': width}, training=False)

EPOCHS_SHORT = 4
CKPT_DIR = ARTIFACTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
ckpt_path = CKPT_DIR / 'exp00_dryrun.weights.h5'

train_ds = build_dataset_bucketed(short_train, LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                  training=True, seed=SEED)
val_ds = build_dataset_bucketed(short_val, LOCAL_DIR, token_to_id, MAX_LABEL_LEN)

history, best_cer = [], float('inf')
for epoch in range(1, EPOCHS_SHORT + 1):
    t0, losses, n_img = time.time(), [], 0
    for b in train_ds:
        losses.append(float(train_step(b['image'], b['width'], b['label'], b['label_length'])))
        n_img += int(b['image'].shape[0])
    train_time = time.time() - t0

    refs, hyps = predict(val_ds)
    m, _ = evaluate_sequences(refs, hyps, families)

    sec_step = train_time / len(losses)
    est_full_min = sec_step * (len(parts['train']) / (n_img / len(losses))) / 60
    history.append({'epoch': epoch, 'train_loss': np.mean(losses), 'val_cer': m['cer'],
                    'val_seq_acc': m['seq_acc'], 'val_cer_shape': m['cer_shape'],
                    'steps': len(losses), 'sec_per_step': sec_step, 'epoch_sec': train_time})
    print(f"epoch {epoch} | loss {np.mean(losses):8.2f} | val CER {m['cer']:.4f} | "
          f"seq_acc {m['seq_acc']:.3f} | cer_shape {m['cer_shape']:.4f} | "
          f"{len(losses)} steps, {sec_step:.2f} s/step | est. full epoch: {est_full_min:.0f} min")

    if m['cer'] < best_cer:
        best_cer = m['cer']
        model.save_weights(ckpt_path)

pd.DataFrame(history).to_csv(ARTIFACTS_DIR / 'exp00_dryrun_history.csv', index=False)

epoch 1 | loss   134.76 | val CER 0.9904 | seq_acc 0.000 | cer_shape 0.9903 | 342 steps, 1.27 s/step | est. full epoch: 84 min
epoch 2 | loss    75.08 | val CER 0.6137 | seq_acc 0.000 | cer_shape 0.5708 | 342 steps, 0.94 s/step | est. full epoch: 63 min
epoch 3 | loss    27.31 | val CER 0.1576 | seq_acc 0.109 | cer_shape 0.1043 | 342 steps, 0.72 s/step | est. full epoch: 48 min
epoch 4 | loss    14.52 | val CER 0.1220 | seq_acc 0.151 | cer_shape 0.0953 | 342 steps, 0.72 s/step | est. full epoch: 48 min


### 6.4. Step times Breakdown

In [11]:
print('TF', tf.__version__, '| Keras', keras.__version__)

b = next(iter(train_ds.skip(40).take(1)))
x, w, lab, ll = b['image'], b['width'], b['label'], b['label_length']
print('batch image shape:', tuple(x.shape))


@tf.function(reduce_retracing=True)
def step_cnn(x):
    with tf.GradientTape() as tape:
        f = x
        for conv, bn, pool in zip(model.convs, model.bns, model.pools):
            f = pool(tf.nn.relu(bn(conv(f), training=True)))
        loss = tf.reduce_mean(f)
    tape.gradient(loss, model.trainable_variables)
    return loss


@tf.function(reduce_retracing=True)
def step_no_ctc(x, w):
    with tf.GradientTape() as tape:
        logits, _ = model({'image': x, 'width': w}, training=True)
        loss = tf.reduce_mean(tf.square(logits))
    tape.gradient(loss, model.trainable_variables)
    return loss


@tf.function(reduce_retracing=True)
def step_ctc(x, w, lab, ll):
    with tf.GradientTape() as tape:
        logits, ln = model({'image': x, 'width': w}, training=True)
        loss = ctc_loss_fn(lab, ll, logits, ln)
    tape.gradient(loss, model.trainable_variables)
    return loss


def bench(fn, *args, n=15):
    float(fn(*args))                       # warm-up / tracing
    t0 = time.time()
    for _ in range(n):
        float(fn(*args))                   # float() forces GPU sync
    return (time.time() - t0) / n


t_cnn = bench(step_cnn, x)
t_rnn = bench(step_no_ctc, x, w)
t_all = bench(step_ctc, x, w, lab, ll)
print(f'CNN only         : {t_cnn:.3f} s/step')
print(f'CNN+BiLSTM+dense : {t_rnn:.3f} s/step  (BiLSTM+dense adds {t_rnn - t_cnn:.3f})')
print(f'+ CTC loss       : {t_all:.3f} s/step  (CTC adds {t_all - t_rnn:.3f})')
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

TF 2.21.0 | Keras 3.13.2
batch image shape: (18, 128, 1266, 1)
CNN only         : 0.070 s/step
CNN+BiLSTM+dense : 0.076 s/step  (BiLSTM+dense adds 0.007)
+ CTC loss       : 0.306 s/step  (CTC adds 0.230)
utilization.gpu [%], memory.used [MiB], memory.total [MiB]
55 %, 12449 MiB, 15360 MiB


6.5. CTC Variants

In [13]:
@tf.function(reduce_retracing=True)
def step_ctc_blank0(x, w, lab, ll):
    with tf.GradientTape() as tape:
        logits, ln = model({'image': x, 'width': w}, training=True)
        logits0 = tf.concat([logits[..., -1:], logits[..., :-1]], axis=-1)  # blank -> index 0
        loss = tf.reduce_mean(tf.nn.ctc_loss(
            labels=lab + 1, logits=tf.cast(logits0, tf.float32),
            label_length=ll, logit_length=ln,
            logits_time_major=False, blank_index=0))
    tape.gradient(loss, model.trainable_variables)
    return loss


@tf.function(reduce_retracing=True)
def step_ctc_blank0_tm(x, w, lab, ll):
    with tf.GradientTape() as tape:
        logits, ln = model({'image': x, 'width': w}, training=True)
        logits0 = tf.concat([logits[..., -1:], logits[..., :-1]], axis=-1)
        loss = tf.reduce_mean(tf.nn.ctc_loss(
            labels=lab + 1, logits=tf.transpose(tf.cast(logits0, tf.float32), [1, 0, 2]),
            label_length=ll, logit_length=ln,
            logits_time_major=True, blank_index=0))
    tape.gradient(loss, model.trainable_variables)
    return loss


t_cur = bench(step_ctc, x, w, lab, ll)
t_b0 = bench(step_ctc_blank0, x, w, lab, ll)
t_b0tm = bench(step_ctc_blank0_tm, x, w, lab, ll)
print(f'blank=last (current)      : {t_cur:.3f} s/step')
print(f'blank=0                   : {t_b0:.3f} s/step')
print(f'blank=0 + time-major      : {t_b0tm:.3f} s/step')

blank=last (current)      : 0.655 s/step
blank=0                   : 0.316 s/step
blank=0 + time-major      : 0.535 s/step


### 6.6. Real Loop

In [14]:
it = iter(train_ds)
wait = step = 0.0
N_TIMED = 50
for i in range(10 + N_TIMED):
    t0 = time.time(); b = next(it); t1 = time.time()
    float(train_step(b['image'], b['width'], b['label'], b['label_length'])); t2 = time.time()
    if i >= 10:                      # skip warm-up
        wait += t1 - t0; step += t2 - t1
print(f'data wait {wait/N_TIMED:.3f} s/step | train_step {step/N_TIMED:.3f} s/step')

data wait 0.001 s/step | train_step 0.785 s/step
